# 运行说明
在第 2 个代码单元底部将 `DATABASE_PATH` 和 `OUTPUT_ROOT` 设为绝对路径。
同时可自定义：`TEMPERATURES`（温度列表）、`WEIGHTS_BY_METHOD`（按方法的权重绝对路径列表）、`BATCH_SIZE`、`NUMBER_OF_BATCHES`、`SAVE_PDB`（0/1）。
若 `DATABASE_PATH` / `OUTPUT_ROOT` / `DESIGN_CHAINS` 为空会直接报错并停止。
当某个子 dataset 缺少 `pipeline.list` 时，会自动扫描该子目录下所有 `.pdb` 文件并创建 `pipeline.list`。

In [1]:
import json
import os
import re
import subprocess
import time
from datetime import datetime
from itertools import product
from pathlib import Path

import pandas as pd


class Colors:
    HEADER = "\033[95m"
    OKBLUE = "\033[94m"
    OKGREEN = "\033[92m"
    WARNING = "\033[93m"
    FAIL = "\033[91m"
    ENDC = "\033[0m"


def log(message, level="INFO"):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if level == "INFO":
        print(f"[{timestamp}] {Colors.OKBLUE}[INFO]{Colors.ENDC} {message}")
    elif level == "SUCCESS":
        print(f"[{timestamp}] {Colors.OKGREEN}[SUCCESS]{Colors.ENDC} {message}")
    elif level == "WARNING":
        print(f"[{timestamp}] {Colors.WARNING}[WARNING]{Colors.ENDC} {message}")
    elif level == "ERROR":
        print(f"[{timestamp}] {Colors.FAIL}[ERROR]{Colors.ENDC} {message}")


def fail_or_warn(message, strict_mode=True):
    if strict_mode:
        log(message, "ERROR")
        raise RuntimeError(message)
    log(message, "WARNING")
    return False


def normalize_temp(temp_value):
    return (f"{float(temp_value):.4f}").rstrip("0").rstrip(".")


def safe_ckpt_name(ckpt_path):
    return Path(ckpt_path).stem.replace("/", "_")


def run_step(script_path, args, step_description):
    log(f"开始执行步骤: {step_description}")
    log("指令: " + " ".join(["bash", script_path] + args))

    if not os.path.exists(script_path):
        raise FileNotFoundError(f"找不到脚本文件: {script_path}")

    start_time = time.time()
    subprocess.run(["bash", script_path] + args, check=True, text=True, capture_output=False)
    elapsed_time = time.time() - start_time
    log(f"步骤 '{step_description}' 完成。耗时: {elapsed_time:.2f} 秒", "SUCCESS")
    return elapsed_time


def read_pipeline_list(pipeline_list_path):
    with open(pipeline_list_path, "r") as handle:
        raw = [line.strip() for line in handle if line.strip() and not line.strip().startswith("#")]
    return raw


def write_jobs_json(pdb_paths, json_path):
    payload = {p: "" for p in pdb_paths}
    with open(json_path, "w") as handle:
        json.dump(payload, handle, indent=2)


def extract_seq_rec_from_fa(fa_path):
    values = []
    with open(fa_path, "r") as handle:
        for line in handle:
            match = re.search(r"seq_rec=(\d+\.\d+)", line)
            if match:
                values.append(round(float(match.group(1)) * 100, 2))
    return values


def collect_seqrec_for_combo(combo_output_dir, target_pdb_set, dataset_name, method, temp, ckpt):
    rows = []
    combo_dir = Path(combo_output_dir)
    if not combo_dir.exists():
        return rows

    for fa_path in combo_dir.rglob("*.fa"):
        pdb_name = fa_path.stem
        if pdb_name not in target_pdb_set:
            continue
        seq_values = extract_seq_rec_from_fa(fa_path)
        for seq_value in seq_values:
            rows.append(
                {
                    "dataset": dataset_name,
                    "method": method,
                    "temperature": temp,
                    "checkpoint": ckpt,
                    "pdb": pdb_name,
                    "seq_rec": seq_value,
                    "fa_path": str(fa_path),
                }
            )
    return rows


def find_existing_pipeline_list_file(dataset_dir: Path):
    pipeline_list = dataset_dir / "pipeline.list"
    if pipeline_list.exists():
        return pipeline_list
    return None


def create_pipeline_list_if_missing(dataset_dir: Path):
    pipeline_list_path = dataset_dir / "pipeline.list"
    if pipeline_list_path.exists():
        return pipeline_list_path

    pdb_paths = sorted([str(p.resolve()) for p in dataset_dir.rglob("*.pdb") if p.is_file()])
    if len(pdb_paths) == 0:
        return None

    with open(pipeline_list_path, "w") as handle:
        handle.write("\n".join(pdb_paths) + "\n")
    log(f"已自动创建 pipeline.list: {pipeline_list_path}（{len(pdb_paths)} 条）", "SUCCESS")
    return pipeline_list_path


def resolve_or_create_pipeline_list(dataset_dir: Path):
    existing = find_existing_pipeline_list_file(dataset_dir)
    if existing is not None:
        return existing
    return create_pipeline_list_if_missing(dataset_dir)


def build_dataset_configs(database_root):
    dataset_configs = []
    for dataset_dir in sorted(Path(database_root).iterdir()):
        if not dataset_dir.is_dir():
            continue
        dataset_name = dataset_dir.name
        pipeline_list_path = resolve_or_create_pipeline_list(dataset_dir)
        dataset_configs.append(
            {
                "name": dataset_name,
                "dir": str(dataset_dir),
                "pipeline_list": str(pipeline_list_path) if pipeline_list_path else "",
            }
        )
    return dataset_configs


def methods_for_dataset(dataset_name):
    if "HETATM" in dataset_name.upper():
        return ["LigandMPNN-HETATM"]
    return ["LigandMPNN", "ProteinMPNN", "PeptideMPNN"]


def normalize_temperatures(temperatures):
    if not isinstance(temperatures, (list, tuple)) or len(temperatures) == 0:
        raise ValueError("temperatures 必须是非空列表，例如 [0.1, 0.2]")

    normalized = []
    for temp in temperatures:
        try:
            normalized.append(float(temp))
        except (TypeError, ValueError) as exc:
            raise ValueError(f"温度值无效: {temp}") from exc
    return normalized


def normalize_weights_by_method(weights_by_method):
    if not isinstance(weights_by_method, dict) or len(weights_by_method) == 0:
        raise ValueError("weights_by_method 必须是非空字典")

    normalized = {}
    for method_name, ckpt_list in weights_by_method.items():
        if not isinstance(ckpt_list, (list, tuple)) or len(ckpt_list) == 0:
            raise ValueError(f"{method_name} 的权重列表不能为空")

        normalized_ckpts = []
        for ckpt in ckpt_list:
            ckpt_path = Path(str(ckpt)).expanduser().resolve()
            if not ckpt_path.exists():
                raise FileNotFoundError(f"checkpoint 不存在: {ckpt_path}")
            normalized_ckpts.append(str(ckpt_path))
        normalized[method_name] = normalized_ckpts

    return normalized


def validate_sampling_config(batch_size, number_of_batches):
    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError("batch_size 必须是正整数")
    if not isinstance(number_of_batches, int) or number_of_batches <= 0:
        raise ValueError("number_of_batches 必须是正整数")


def validate_save_pdb_config(save_pdb):
    if isinstance(save_pdb, bool):
        return int(save_pdb)

    if isinstance(save_pdb, int) and save_pdb in (0, 1):
        return save_pdb

    if isinstance(save_pdb, str) and save_pdb.strip() in {"0", "1"}:
        return int(save_pdb.strip())

    raise ValueError("save_pdb 必须是 0 或 1")


def main(
    database_path,
    output_root,
    chains_to_design,
    temperatures,
    weights_by_method,
    batch_size,
    number_of_batches,
    save_pdb=0,
    dry_run=False,
):
    log("=== 启动多数据集 Benchmark Pipeline ===")

    if database_path is None or str(database_path).strip() == "":
        raise ValueError("必须手动传入 database_path，例如 '/path/to/database'")
    if output_root is None or str(output_root).strip() == "":
        raise ValueError("必须手动传入 output_root，例如 '/path/to/outputs'")
    if chains_to_design is None or str(chains_to_design).strip() == "":
        raise ValueError("必须手动传入 chains_to_design，例如 'L' 或 'A,B'")

    workspace_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    database_root = Path(database_path).expanduser().resolve()
    output_base = Path(output_root).expanduser().resolve()
    output_base.mkdir(parents=True, exist_ok=True)
    chains_to_design = str(chains_to_design).strip()
    temperatures = normalize_temperatures(temperatures)
    weights_by_method = normalize_weights_by_method(weights_by_method)
    validate_sampling_config(batch_size, number_of_batches)
    save_pdb = validate_save_pdb_config(save_pdb)
    strict_mode = True

    method_scripts = {
        "LigandMPNN": str(workspace_dir / "benchmark_ligandmpnn.sh"),
        "LigandMPNN-HETATM": str(workspace_dir / "benchmark_ligandmpnn.sh"),
        "ProteinMPNN": str(workspace_dir / "benchmark_proteinmpnn.sh"),
        "PeptideMPNN": str(workspace_dir / "benchmark_proteinmpnn.sh"),
    }

    if not database_root.exists():
        raise FileNotFoundError(f"database目录不存在: {database_root}")

    dataset_configs = build_dataset_configs(database_root)
    if len(dataset_configs) == 0:
        raise RuntimeError(f"database目录下没有可用数据集子目录: {database_root}")

    if dry_run:
        log("当前为 DRY_RUN 模式：仅生成 pipeline.list / jobs.json，并打印执行计划，不调用 MPNN", "WARNING")

    run_records = []
    seqrec_rows_all = []

    for dataset in dataset_configs:
        dataset_name = dataset["name"]
        log(f"=== 处理数据集: {dataset_name} ===")

        if not dataset["pipeline_list"]:
            fail_or_warn(f"{dataset_name} 缺少 pipeline.list 且未发现可用于自动生成的 .pdb 文件", strict_mode)
            continue

        pipeline_list_path = Path(dataset["pipeline_list"])
        if not pipeline_list_path.exists():
            fail_or_warn(f"缺少 pipeline.list: {pipeline_list_path}", strict_mode)
            continue

        pdb_abs_paths = read_pipeline_list(pipeline_list_path)
        if len(pdb_abs_paths) == 0:
            fail_or_warn(f"pipeline.list 为空: {pipeline_list_path}", strict_mode)
            continue

        missing_paths = [p for p in pdb_abs_paths if not os.path.isabs(p) or not Path(p).exists()]
        if len(missing_paths) > 0:
            fail_or_warn(f"pipeline.list 中存在无效路径，示例: {missing_paths[:3]}", strict_mode)
            continue

        target_pdb_set = {Path(p).stem for p in pdb_abs_paths}

        dataset_out_root = output_base / dataset_name
        dataset_out_root.mkdir(parents=True, exist_ok=True)

        jobs_json_path = dataset_out_root / f"{dataset_name}.json"
        write_jobs_json(pdb_abs_paths, jobs_json_path)

        enabled_methods = methods_for_dataset(dataset_name)
        log(f"数据集 {dataset_name} 运行方法: {enabled_methods}")

        for method in enabled_methods:
            method_script = method_scripts[method]
            input_json = str(jobs_json_path)

            if method not in weights_by_method or len(weights_by_method[method]) == 0:
                fail_or_warn(f"方法 {method} 未配置权重", strict_mode)
                continue

            for temp, ckpt in product(temperatures, weights_by_method[method]):
                temp_str = normalize_temp(temp)
                ckpt_name = safe_ckpt_name(ckpt)
                combo_name = f"temp_{temp_str}_ckpt_{ckpt_name}"
                combo_out_dir = dataset_out_root / method / combo_name

                record = {
                    "dataset": dataset_name,
                    "method": method,
                    "temperature": temp_str,
                    "checkpoint": ckpt,
                    "chains_to_design": chains_to_design,
                    "batch_size": batch_size,
                    "number_of_batches": number_of_batches,
                    "save_pdb": save_pdb,
                    "total_samples": batch_size * number_of_batches,
                    "input_json": input_json,
                    "output_dir": str(combo_out_dir),
                    "status": "FAILED",
                    "elapsed_sec": None,
                    "error": "",
                }

                if dry_run:
                    record["status"] = "DRY_RUN"
                    log(
                        f"[DRY_RUN] 计划执行: dataset={dataset_name}, method={method}, temp={temp_str}, ckpt={ckpt}, batch_size={batch_size}, number_of_batches={number_of_batches}, save_pdb={save_pdb}, input_json={input_json}",
                        "INFO",
                    )
                    run_records.append(record)
                    continue

                try:
                    elapsed = run_step(
                        script_path=method_script,
                        args=[
                            input_json,
                            str(combo_out_dir),
                            str(temp),
                            ckpt,
                            chains_to_design,
                            str(batch_size),
                            str(number_of_batches),
                            str(save_pdb),
                        ],
                        step_description=f"{dataset_name} | {method} | {combo_name}",
                    )
                    record["status"] = "SUCCESS"
                    record["elapsed_sec"] = round(elapsed, 2)

                    seq_rows = collect_seqrec_for_combo(
                        combo_output_dir=combo_out_dir,
                        target_pdb_set=target_pdb_set,
                        dataset_name=dataset_name,
                        method=method,
                        temp=temp_str,
                        ckpt=ckpt,
                    )
                    if len(seq_rows) == 0:
                        msg = f"未找到有效seq_rec记录: {dataset_name}/{method}/{combo_name}"
                        if strict_mode:
                            raise RuntimeError(msg)
                        log(msg, "WARNING")
                    seqrec_rows_all.extend(seq_rows)

                except Exception as exc:
                    record["error"] = str(exc)
                    log(
                        f"组合失败: dataset={dataset_name}, method={method}, temp={temp_str}, ckpt={ckpt} | {exc}",
                        "ERROR",
                    )
                    run_records.append(record)
                    if strict_mode:
                        run_df = pd.DataFrame(run_records)
                        run_df.to_csv(dataset_out_root / "run_metadata.csv", index=False)
                        raise
                    continue

                run_records.append(record)

        run_df = pd.DataFrame([r for r in run_records if r["dataset"] == dataset_name])
        run_df.to_csv(dataset_out_root / "run_metadata.csv", index=False)

        if dry_run:
            log(f"数据集 {dataset_name} 在 DRY_RUN 模式下跳过 seq_rec 统计", "WARNING")
            continue

        dataset_seq_rows = [r for r in seqrec_rows_all if r["dataset"] == dataset_name]
        if len(dataset_seq_rows) > 0:
            details_df = pd.DataFrame(dataset_seq_rows)
            details_path = dataset_out_root / "seqrec_details.csv"
            details_df.to_csv(details_path, index=False)

            summary_df = (
                details_df.groupby(["dataset", "method", "temperature", "checkpoint"], as_index=False)["seq_rec"]
                .agg(["count", "mean", "std", "min", "max"])
                .reset_index()
            )
            summary_path = dataset_out_root / "seqrec_summary.csv"
            summary_df.to_csv(summary_path, index=False)
            log(f"已写出统计结果: {summary_path}", "SUCCESS")
        else:
            log(f"数据集 {dataset_name} 无可用 seq_rec 记录", "WARNING")

    all_run_df = pd.DataFrame(run_records)
    all_run_path = output_base / "run_metadata_all.csv"
    all_run_df.to_csv(all_run_path, index=False)

    log(f"全局运行记录已写出: {all_run_path}", "SUCCESS")
    log("=== Pipeline 执行结束 ===", "SUCCESS")


BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
MODEL_DIR = BASE_DIR / "model_params"

DATABASE_PATH = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/02_ligandmpnn/test_hyperparam/src/test"
OUTPUT_ROOT = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/02_ligandmpnn/test_hyperparam/test"
DESIGN_CHAINS = "B"
TEMPERATURES = [0.1]
WEIGHTS_BY_METHOD = {
    "LigandMPNN": [
        str(MODEL_DIR / "ligandmpnn_v_32_005_25.pt"),
        str(MODEL_DIR / "ligandmpnn_v_32_010_25.pt"),
        str(MODEL_DIR / "ligandmpnn_v_32_020_25.pt"),
        str(MODEL_DIR / "ligandmpnn_v_32_030_25.pt"),
    # ],
    # "LigandMPNN-HETATM": [
    #     str(MODEL_DIR / "ligandmpnn_v_32_005_25.pt"),
    #     str(MODEL_DIR / "ligandmpnn_v_32_010_25.pt"),
    #     str(MODEL_DIR / "ligandmpnn_v_32_020_25.pt"),
    #     str(MODEL_DIR / "ligandmpnn_v_32_030_25.pt"),
    # ],
    # "ProteinMPNN": [
    #     str(MODEL_DIR / "proteinmpnn_v_48_002.pt"),
    #     str(MODEL_DIR / "proteinmpnn_v_48_010.pt"),
    #     str(MODEL_DIR / "proteinmpnn_v_48_020.pt"),
    #     str(MODEL_DIR / "proteinmpnn_v_48_030.pt"),
    # ],
    # "PeptideMPNN": [
    #     str(BASE_DIR / "PeptideMPNN" / "epoch_last.pt"),
    ],
}
BATCH_SIZE = 10
NUMBER_OF_BATCHES = 1
SAVE_PDB = 0
DRY_RUN = False

main(
    database_path=DATABASE_PATH,
    output_root=OUTPUT_ROOT,
    chains_to_design=DESIGN_CHAINS,
    temperatures=TEMPERATURES,
    weights_by_method=WEIGHTS_BY_METHOD,
    batch_size=BATCH_SIZE,
    number_of_batches=NUMBER_OF_BATCHES,
    save_pdb=SAVE_PDB,
    dry_run=DRY_RUN,
)

[2026-06-02 15:26:17] [INFO] === 启动多数据集 Benchmark Pipeline ===
[2026-06-02 15:26:17] [INFO] === 处理数据集: Merged_PDBs ===
[2026-06-02 15:26:17] [INFO] 数据集 Merged_PDBs 运行方法: ['LigandMPNN', 'ProteinMPNN', 'PeptideMPNN']
[2026-06-02 15:26:17] [INFO] 开始执行步骤: Merged_PDBs | LigandMPNN | temp_0.1_ckpt_ligandmpnn_v_32_005_25
[2026-06-02 15:26:17] [INFO] 指令: bash /QIN/junjiechen/250401-Dpepalign/Benchmark/02_ligandmpnn/test_hyperparam/scripts/benchmark_ligandmpnn.sh /QIN/junjiechen/250401-Dpepalign/Benchmark/02_ligandmpnn/test_hyperparam/test/Merged_PDBs/Merged_PDBs.json /QIN/junjiechen/250401-Dpepalign/Benchmark/02_ligandmpnn/test_hyperparam/test/Merged_PDBs/LigandMPNN/temp_0.1_ckpt_ligandmpnn_v_32_005_25 0.1 /QIN/junjiechen/original_soft/LigandMPNN/model_params/ligandmpnn_v_32_005_25.pt B 10 1 0
[INFO] Running LigandMPNN
[INFO] pdb_json=/QIN/junjiechen/250401-Dpepalign/Benchmark/02_ligandmpnn/test_hyperparam/test/Merged_PDBs/Merged_PDBs.json
[INFO] out_dir=/QIN/junjiechen/250401-Dpepalign/Benchm

Traceback (most recent call last):
  File "/home/junjiechen/1_work/250401-Dpepalign/soft/LigandMPNN/run_test.py", line 1005, in <module>
    main(args)
  File "/home/junjiechen/1_work/250401-Dpepalign/soft/LigandMPNN/run_test.py", line 434, in main
    output_dict = model.sample(feature_dict)
  File "/QIN/junjiechen/250401-Dpepalign/soft/LigandMPNN/model_utils.py", line 301, in sample
    t[:, None, None].repeat(1, 1, h_V_stack[l].shape[-1]),
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.family"] = "Arial"

METHODS = ["LigandMPNN", "LigandMPNN-HETATM", "ProteinMPNN", "PeptideMPNN"]
PALETTE = ["black", "red", "blue", "orange", "green"]

# 开关：None 表示全部；也可填单个字符串或字符串列表
SELECT_METHODS = None
SELECT_TEMPERATURES = None


def normalize_temp_for_plot(value):
    return (f"{float(value):.4f}").rstrip("0").rstrip(".")


def extract_pass_value(dataset_name):
    dataset_lower = dataset_name.lower()
    pass_index = dataset_lower.find("pass")
    if pass_index < 0:
        return None

    tail = dataset_lower[pass_index:]
    numbers = re.findall(r"\d+(?:\.\d+)?", tail)
    if len(numbers) == 0:
        return 0.0
    return float(numbers[-1])


def normalize_selector(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float)):
        return {str(value)}
    if isinstance(value, (list, tuple, set)):
        return {str(v) for v in value}
    raise ValueError("选择器仅支持 None/字符串/列表")


def method_matches(method_value, method_selector_set):
    if method_selector_set is None:
        return True
    lowered = {v.lower() for v in method_selector_set}
    return str(method_value).lower() in lowered


plot_root = Path(OUTPUT_ROOT) if "OUTPUT_ROOT" in globals() else Path.cwd() / "outputs"
plot_dir = plot_root / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

summary_paths = sorted(plot_root.glob("*/seqrec_summary.csv"))
if not summary_paths:
    raise FileNotFoundError(f"在 {plot_root} 下未找到 seqrec_summary.csv")

frames = []
for csv_path in summary_paths:
    dataset_name = csv_path.parent.name
    noise = extract_pass_value(dataset_name)
    if noise is None or noise > 0.5:
        continue

    df = pd.read_csv(csv_path)
    required_cols = {"method", "temperature", "checkpoint", "mean"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"{csv_path} 缺少列: {sorted(missing_cols)}")

    work_df = df.copy()
    work_df["method"] = work_df["method"].astype(str).str.strip()
    work_df["noise"] = noise
    work_df["temp_key"] = work_df["temperature"].apply(normalize_temp_for_plot)
    work_df["checkpoint_name"] = work_df["checkpoint"].apply(lambda x: Path(str(x)).stem)
    work_df["dataset_name"] = dataset_name
    frames.append(work_df[["dataset_name", "method", "temp_key", "checkpoint_name", "noise", "mean"]])

if not frames:
    raise RuntimeError("没有 pass 到 pass-0.5 的可绘图数据")

plot_df = pd.concat(frames, ignore_index=True)
plot_df["mean"] = pd.to_numeric(plot_df["mean"], errors="coerce")
plot_df = plot_df.dropna(subset=["mean"])

selected_methods = normalize_selector(SELECT_METHODS)
selected_temps = normalize_selector(SELECT_TEMPERATURES)

plot_df = plot_df[plot_df["method"].apply(lambda m: method_matches(m, selected_methods))]
if selected_temps is not None:
    plot_df = plot_df[plot_df["temp_key"].astype(str).isin(selected_temps)]

if plot_df.empty:
    raise RuntimeError("筛选后无可绘图数据，请检查 SELECT_METHODS / SELECT_TEMPERATURES")

method_order = [m for m in METHODS if m in set(plot_df["method"].unique())]
temp_order = sorted(plot_df["temp_key"].dropna().unique(), key=lambda x: float(x))

coverage_df = (
    plot_df.groupby(["method", "temp_key"], as_index=False)["noise"]
    .apply(lambda s: sorted({round(float(v), 3) for v in s}))
    .rename(columns={"noise": "noise_points"})
)
print("=== 数据覆盖检查（method / temperature / noise）===")
print(coverage_df.to_string(index=False))

# 图组1：固定方法 + 固定温度，不同 checkpoint
count_fixed = 0
for method_name in method_order:
    for temp_key in temp_order:
        subset = plot_df[(plot_df["method"] == method_name) & (plot_df["temp_key"] == temp_key)].copy()
        if subset.empty:
            continue

        plt.figure(figsize=(8, 5))

        checkpoint_order = sorted(subset["checkpoint_name"].dropna().unique())
        for idx, checkpoint_name in enumerate(checkpoint_order):
            group_df = subset[subset["checkpoint_name"] == checkpoint_name]
            group_df = (
                group_df.groupby("noise", as_index=False)["mean"]
                .mean()
                .sort_values("noise")
            )
            color = PALETTE[idx % len(PALETTE)]
            plt.plot(
                group_df["noise"],
                group_df["mean"],
                marker="o",
                linewidth=2,
                color=color,
                label=checkpoint_name,
            )

        noise_ticks = sorted(subset["noise"].dropna().unique())
        noise_labels = [str(int(x)) if float(x).is_integer() else str(x) for x in noise_ticks]
        plt.xticks(noise_ticks, noise_labels)
        plt.xlabel("PDB Noise (Å)")
        plt.ylabel("seq. rec (%)")
        plt.ylim(0, 100)
        plt.title(f"{method_name} | temperature={temp_key}")
        plt.grid(alpha=0.25, linestyle="--")
        plt.legend(title="Checkpoint", loc="best")
        plt.tight_layout()

        method_slug = method_name.replace("/", "_").replace(" ", "_")
        out_png = plot_dir / f"seqrec_mean_{method_slug}_temp_{temp_key}.png"
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"[PLOT-固定温度] 已保存: {out_png}")
        count_fixed += 1

# 图组2：固定方法，不同温度（每条线=一个温度，先对 checkpoint 取均值）
count_temp_compare = 0
for method_name in method_order:
    subset = plot_df[plot_df["method"] == method_name].copy()
    if subset.empty:
        continue

    plt.figure(figsize=(8, 5))

    temp_keys = sorted(subset["temp_key"].dropna().unique(), key=lambda x: float(x))
    for idx, temp_key in enumerate(temp_keys):
        group_df = subset[subset["temp_key"] == temp_key]
        group_df = (
            group_df.groupby("noise", as_index=False)["mean"]
            .mean()
            .sort_values("noise")
        )
        color = PALETTE[idx % len(PALETTE)]
        plt.plot(
            group_df["noise"],
            group_df["mean"],
            marker="o",
            linewidth=2,
            color=color,
            label=f"temp={temp_key}",
        )

    noise_ticks = sorted(subset["noise"].dropna().unique())
    noise_labels = [str(int(x)) if float(x).is_integer() else str(x) for x in noise_ticks]
    plt.xticks(noise_ticks, noise_labels)
    plt.xlabel("PDB Noise (Å)")
    plt.ylabel("seq. rec (%)")
    plt.ylim(0, 100)
    plt.title(f"{method_name} | temperature compare")
    plt.grid(alpha=0.25, linestyle="--")
    plt.legend(title="Temperature", loc="best")
    plt.tight_layout()

    method_slug = method_name.replace("/", "_").replace(" ", "_")
    out_png = plot_dir / f"seqrec_mean_{method_slug}_temperature_compare.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"[PLOT-温度对比] 已保存: {out_png}")
    count_temp_compare += 1

# 图组3：固定权重( checkpoint ) + 固定温度，不同方法
count_method_compare = 0
for temp_key in temp_order:
    for checkpoint_name in sorted(plot_df["checkpoint_name"].dropna().unique()):
        subset = plot_df[(plot_df["temp_key"] == temp_key) & (plot_df["checkpoint_name"] == checkpoint_name)].copy()
        if subset.empty:
            continue

        method_present = [m for m in METHODS if m in set(subset["method"].unique())]
        if len(method_present) == 0:
            continue

        plt.figure(figsize=(8, 5))

        for idx, method_name in enumerate(method_present):
            group_df = subset[subset["method"] == method_name]
            group_df = (
                group_df.groupby("noise", as_index=False)["mean"]
                .mean()
                .sort_values("noise")
            )
            if group_df.empty:
                continue

            color = PALETTE[idx % len(PALETTE)]
            plt.plot(
                group_df["noise"],
                group_df["mean"],
                marker="o",
                linewidth=2,
                color=color,
                label=method_name,
            )

        noise_ticks = sorted(subset["noise"].dropna().unique())
        noise_labels = [str(int(x)) if float(x).is_integer() else str(x) for x in noise_ticks]
        plt.xticks(noise_ticks, noise_labels)
        plt.xlabel("PDB Noise (Å)")
        plt.ylabel("seq. rec (%)")
        plt.ylim(0, 100)
        plt.title(f"checkpoint={checkpoint_name} | temperature={temp_key} | method compare")
        plt.grid(alpha=0.25, linestyle="--")
        plt.legend(title="Method", loc="best")
        plt.tight_layout()

        ckpt_slug = checkpoint_name.replace("/", "_").replace(" ", "_")
        out_png = plot_dir / f"seqrec_mean_method_compare_ckpt_{ckpt_slug}_temp_{temp_key}.png"
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"[PLOT-方法对比] 已保存: {out_png}")
        count_method_compare += 1

print(
    f"绘图完成：固定温度图 {count_fixed} 张，温度对比图 {count_temp_compare} 张，方法对比图 {count_method_compare} 张；保存目录: {plot_dir}"
)

In [3]:
import re
from pathlib import Path

import pandas as pd

SUMMARY_FILENAME = "seqrec_summary.csv"
TEMPERATURE = 0.1
OUTPUT_CSV = "seqrec_summary_temp_0.1_matrix.csv"

METHOD_ORDER = ["LigandMPNN", "LigandMPNN-HETATM", "ProteinMPNN"]


def normalize_temp(value):
    return (f"{float(value):.4f}").rstrip("0").rstrip(".")


def extract_noise(dataset_name: str):
    dataset_lower = dataset_name.lower()
    pass_idx = dataset_lower.find("pass")
    if pass_idx < 0:
        return None
    tail = dataset_lower[pass_idx:]
    numbers = re.findall(r"\d+(?:\.\d+)?", tail)
    if not numbers:
        return 0.0
    return float(numbers[-1])


def checkpoint_label(checkpoint_value: str):
    stem = Path(str(checkpoint_value)).stem
    chunks = re.findall(r"\d+", stem)
    three_digit = [c for c in chunks if len(c) == 3]
    if three_digit:
        return str(int(three_digit[-1]))
    if chunks:
        return str(int(chunks[-1]))
    return stem


def method_sort_key(method_name: str):
    if method_name in METHOD_ORDER:
        return METHOD_ORDER.index(method_name)
    return len(METHOD_ORDER)


root = Path(OUTPUT_ROOT) if "OUTPUT_ROOT" in globals() else Path.cwd() / "outputs"
summary_paths = sorted(root.glob(f"*/{SUMMARY_FILENAME}"))
if not summary_paths:
    raise FileNotFoundError(f"在 {root} 下未找到 {SUMMARY_FILENAME}")

target_temp = normalize_temp(TEMPERATURE)
all_rows = []
all_combos = set()

for csv_path in summary_paths:
    dataset_name = csv_path.parent.name
    noise = extract_noise(dataset_name)

    df = pd.read_csv(csv_path)
    required = {"method", "temperature", "checkpoint", "mean", "std"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} 缺少列: {sorted(missing)}")

    work = df.copy()
    work["temp_key"] = work["temperature"].apply(normalize_temp)
    work = work[work["temp_key"] == target_temp]
    if work.empty:
        continue

    row = {"dataset": dataset_name, "noise": noise}
    for _, record in work.iterrows():
        method = str(record["method"]).strip()
        ckpt = checkpoint_label(record["checkpoint"])
        combo = f"{method}-{ckpt}"
        all_combos.add((method, ckpt))
        row[f"{combo}-MEAN"] = record["mean"]
        row[f"{combo}-std"] = record["std"]

    all_rows.append(row)

if not all_rows:
    raise RuntimeError(f"没有找到温度={target_temp} 的可用统计数据")

result_df = pd.DataFrame(all_rows)

combo_order = sorted(
    all_combos,
    key=lambda x: (
        method_sort_key(x[0]),
        0 if str(x[1]).isdigit() else 1,
        int(x[1]) if str(x[1]).isdigit() else str(x[1]),
    ),
)

ordered_cols = ["dataset", "noise"]
for method, ckpt in combo_order:
    combo = f"{method}-{ckpt}"
    mean_col = f"{combo}-MEAN"
    std_col = f"{combo}-std"
    if mean_col in result_df.columns:
        ordered_cols.append(mean_col)
    if std_col in result_df.columns:
        ordered_cols.append(std_col)

result_df = result_df.reindex(columns=ordered_cols)
result_df = result_df.sort_values(by=["noise", "dataset"], na_position="last")

out_path = root / OUTPUT_CSV
result_df.to_csv(out_path, index=False)
print(f"已导出: {out_path}")
print(result_df.head().to_string(index=False))

已导出: /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_hyperparam/outputs/seqrec_summary_temp_0.1_matrix.csv
                       dataset  noise  LigandMPNN-5-MEAN  LigandMPNN-5-std  LigandMPNN-10-MEAN  LigandMPNN-10-std  LigandMPNN-20-MEAN  LigandMPNN-20-std  LigandMPNN-30-MEAN  LigandMPNN-30-std  LigandMPNN-HETATM-5-MEAN  LigandMPNN-HETATM-5-std  LigandMPNN-HETATM-10-MEAN  LigandMPNN-HETATM-10-std  LigandMPNN-HETATM-20-MEAN  LigandMPNN-HETATM-20-std  LigandMPNN-HETATM-30-MEAN  LigandMPNN-HETATM-30-std  ProteinMPNN-2-MEAN  ProteinMPNN-2-std  ProteinMPNN-10-MEAN  ProteinMPNN-10-std  ProteinMPNN-20-MEAN  ProteinMPNN-20-std  ProteinMPNN-30-MEAN  ProteinMPNN-30-std
           PepSet_AF3_noC_pass    0.0          53.228020         17.627503           52.397373          17.166373           49.374373          17.363748           46.744882          19.028638                       NaN                      NaN                        NaN                       NaN               